# 06 (v2.1). 문장형 답변 LoRA

팀 목표인 **"답할 때는 완전한 문장으로"**에 맞춰, MS MARCO v2.1의 **문장형 정답(wellFormedAnswers)**으로 LoRA를 다시 학습합니다.
05(v1.1)는 짧은 정답 구절로 학습해서 답이 14단어 정도로 짧아졌습니다. v1.1에는 문장형 정답이 없어서 이번에는 v2.1을 씁니다.

```
v2.1 train 건강 질문 (문장형 정답 있음) ─▶ v2.1 DB에서 상위 5개 문단 ─▶ "문장형 정답 [번호]"
v2.1 train 건강 질문 (No Answer Present) ─▶ v2.1 DB에서 상위 5개 문단 ─▶ "거절 문장"
                                          ─▶ Qwen2.5-7B-Instruct LoRA (프롬프트는 04·05와 같음)
```

| 단계 | 내용 |
|---|---|
| 3. 검색기 | v2.1 의료 DB (문단 459,067, 드라이브 `health/`). bge → MiniLM-L-6 재정렬, 중복 제거 |
| 4. 거절 기준 | v2.1 검증 200개의 재정렬 최고 점수 하위 5% |
| 5. 학습 데이터 | 문단이 DB에 있는 train 질문만 사용 (문장형 정답 4,785개 중 `N_TRAIN`개 + 거절 예시 15%) |
| 6. LoRA 학습 | 05와 같은 설정 (bf16 LoRA r=16, 1 epoch) |
| 7. 평가 | **원본 Qwen** vs **짧은 답 LoRA (05)** vs **문장형 LoRA**. 문장형 정답이 있는 건강 107 + 답 없음 100 + 분야 밖 50 |

**지표**
- **ROUGE-L (문장형)**: 문장형 정답과의 겹침 — 이번 실험의 핵심 지표
- **정답 포함률**: 짧은 정답의 단어가 답변 안에 얼마나 들어 있는지 (문장으로 길게 답해도 깎이지 않음)
- F1 (짧은 정답), 출처 표시율, 잘못 거절, **답 없음 거절률**, 분야 밖 거절률, 평균 답변 단어

**런타임: A100 + High-RAM.** 위에서부터 Run all. 전체 약 50분 (DB 불러오기 5~8분, 학습 15분, 평가 15분).
결과는 드라이브 `rag_msmarco/health/lora_sentence/`와 `RAG/2_results/lora_sentence/`에 저장되고, 마지막 셀이 런타임을 끕니다.

## 1. 설치 + 저장 위치

In [ ]:
!pip -q install -U sentence-transformers faiss-cpu peft datasets
!pip -q uninstall -y torchao    # Colab 기본 torchao(0.10)가 최신 peft와 충돌해서 제거

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR    = '/content/drive/MyDrive/rag_msmarco/health'          # v2.1 DB (02에서 만든 것)
OUT_DIR     = f'{SAVE_DIR}/lora_sentence'
REPORT_DIR  = '/content/drive/MyDrive/RAG/2_results/lora_sentence'  # 결과 요약 사본 (Mac RAG 폴더와 동기화)
SHORT_LORA  = '/content/drive/MyDrive/rag_msmarco/v11_test/lora/adapter'   # 05에서 학습한 짧은 답 LoRA
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)
print('DB 위치:', SAVE_DIR, '/ 결과 저장:', OUT_DIR)
print('05 LoRA 있음:', os.path.isdir(SHORT_LORA))

## 2. 설정

시간이 밀리면 `N_TRAIN`부터 줄이세요 (2000이면 학습 약 10분).

In [ ]:
import re, json, gzip, time, random
from collections import Counter
from urllib.parse import urlparse
import numpy as np
import pandas as pd
import torch

SEED       = 42
K          = 100        # 1단계 후보 수 (= 재정렬할 후보 수)
TOP_N      = 5          # LLM에 넣을 문단 수
GEN_BATCH  = 16
MAX_NEW    = 256

N_TRAIN       = 3000    # 학습 예시 수 (거절 예시 포함)
REFUSE_FRAC   = 0.15    # 학습 예시 중 거절 예시 비율
MAX_LEN       = 1024    # 이보다 긴 예시는 버림
EPOCHS        = 1
LR            = 2e-4
BATCH, GRAD_ACC = 8, 2  # 실제 배치 16
LORA_R, LORA_ALPHA = 16, 32

EMB_MODEL  = 'BAAI/bge-base-en-v1.5'
QUERY_PREFIX = 'Represent this sentence for searching relevant passages: '
RERANKER   = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
LLM_NAME   = 'Qwen/Qwen2.5-7B-Instruct'
NO_ANSWER  = 'No Answer Present.'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
assert DEVICE == 'cuda', 'GPU 런타임이 아닙니다. 런타임 유형을 A100으로 바꾸세요.'
print('GPU:', torch.cuda.get_device_name(0), f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.0f}GB')

def load_jsonl(path):
    opener = gzip.open if path.endswith('.gz') else open
    with opener(path, 'rt', encoding='utf-8') as f:
        return [json.loads(line) for line in f]

def save_jsonl(rows, path):
    with open(path, 'w', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')

## 3. 검색기 (v2.1 DB, bge → MiniLM 재정렬)

04·05와 같은 방식입니다. DB가 크고(문단 46만, 인덱스 1.4GB) 드라이브에서 읽느라 불러오기에 5~8분 걸립니다.

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder

t = time.time()
corpus   = load_jsonl(f'{SAVE_DIR}/corpus.jsonl')
eval_set = load_jsonl(f'{SAVE_DIR}/eval.jsonl')
val_set  = load_jsonl(f'{SAVE_DIR}/rerank/val.jsonl')
index    = faiss.read_index(f'{SAVE_DIR}/faiss_bge_flatip.index')
texts    = [c['text'] for c in corpus]
assert index.ntotal == len(corpus), '인덱스와 문단 수가 다릅니다'
print(f'문단 {len(corpus):,} / 평가 {len(eval_set)} / 검증 {len(val_set)} ({time.time() - t:.0f}초)')

emb_model = SentenceTransformer(EMB_MODEL, device=DEVICE)
emb_model.half()
reranker = CrossEncoder(RERANKER, max_length=256, device=DEVICE)
reranker.model.half()

TIERS = {
    '공공기관':   ['medlineplus.gov', 'cdc.gov', 'nih.gov', 'nhs.uk', 'who.int', 'fda.gov', 'cancer.gov'],
    '병원·학회':  ['mayoclinic.org', 'clevelandclinic.org', 'hopkinsmedicine.org', 'aafp.org', 'merckmanuals.com',
                  'kidshealth.org', 'health.harvard.edu', 'heart.org', 'cancer.org', 'familydoctor.org', 'uptodate.com'],
    '상업 매체':  ['webmd.com', 'healthline.com', 'verywell.com', 'medicinenet.com', 'everydayhealth.com', 'drugs.com',
                  'rxlist.com', 'emedicinehealth.com', 'medscape.com', 'medicalnewstoday.com', 'rightdiagnosis.com',
                  'patient.info', 'labtestsonline.org'],
    '커뮤니티·생활': ['healthtap.com', 'sharecare.com', 'livestrong.com', 'babycenter.com', 'psychologytoday.com'],
    '사전':       ['thefreedictionary.com'],
}

def site_of(url):
    h = urlparse(url).netloc.lower()
    return h[4:] if h.startswith('www.') else h

def tier_of(site):
    for tier, sites in TIERS.items():
        if any(site == s or site.endswith('.' + s) for s in sites):
            return tier
    return '기타'

sites = [site_of(c['url']) for c in corpus]
norm_text = lambda t: ' '.join(t.lower().split())

def encode(strings):
    return emb_model.encode([QUERY_PREFIX + s for s in strings], batch_size=128, normalize_embeddings=True,
                            convert_to_numpy=True).astype('float32')

def retrieve_many(queries, top_n=TOP_N):
    # 질문마다 [{pid, score, top_raw, site, tier, text}, ...] 상위 top_n개 (본문이 같은 문단은 하나만)
    _, ids = index.search(encode(queries), K)
    pairs = [(q, texts[p]) for q, cands in zip(queries, ids.tolist()) for p in cands]
    all_scores = reranker.predict(pairs, batch_size=512, show_progress_bar=len(queries) > 500).reshape(len(queries), K)
    out = []
    for cands, scores in zip(ids.tolist(), all_scores):
        docs, seen = [], set()
        for i in np.argsort(-scores):
            p = int(cands[i])
            key = norm_text(texts[p])
            if key in seen:
                continue
            seen.add(key)
            docs.append({'pid': p, 'score': float(scores[i]), 'top_raw': float(scores.max()),
                         'site': sites[p], 'tier': tier_of(sites[p]), 'text': texts[p]})
            if len(docs) == top_n:
                break
        out.append(docs)
    return out

## 4. 거절 기준 (검증 질문으로만)

04와 같은 규칙: 검증 200개(모두 답 있는 건강 질문)의 재정렬 최고 점수 하위 5%보다 낮으면 LLM을 부르지 않고 거절합니다.

In [ ]:
val_docs = retrieve_many([q['query'] for q in val_set])
THRESH = float(np.percentile([d[0]['top_raw'] for d in val_docs], 5))
val_r5 = np.mean([bool(set(q['gold_pids']) & {d['pid'] for d in docs}) for q, docs in zip(val_set, val_docs)])
print(f'거절 기준: {THRESH:.2f} / 검증 근거 R@5: {val_r5:.3f}')

## 5. 학습 데이터 (v2.1 train 건강 질문)

- `train_health_qa.jsonl.gz` (02에서 만든 것) 중 **문단이 DB에 들어 있는 질문**만 씁니다. 그래야 검색 결과에 정답 문단이 나올 수 있습니다.
- **답변 예시**: 문장형 정답이 있고, 정답 문단이 상위 5개 안에 있으면 → `문장형 정답 [번호]`
- **거절 예시**: 정답이 `No Answer Present.`인 질문 → 거절 문장 (모자라면 정답 문단이 상위 5개 밖인 질문으로 채움)
- 평가·검증 질문은 dev에서 왔으므로 train과 겹치지 않습니다.

In [ ]:
REFUSAL = "Sorry, I can't answer that from the medical documents I have."

pid_of = {t: i for i, t in enumerate(texts)}
held_out = {q['qid'] for q in eval_set} | {q['qid'] for q in val_set}

wf_pool, noans_pool = [], []
with gzip.open(f'{SAVE_DIR}/train_health_qa.jsonl.gz', 'rt', encoding='utf-8') as f:
    for line in f:
        q = json.loads(line)
        if not q['in_corpus'] or q['qid'] in held_out:
            continue
        gold = {pid_of[p['text']] for p in q['passages'] if p['is_selected'] == 1 and p['text'] in pid_of}
        if q['answers'] == [NO_ANSWER]:
            noans_pool.append({'qid': q['qid'], 'query': q['query']})
        elif q['well_formed'] and gold:
            wf_pool.append({'qid': q['qid'], 'query': q['query'], 'answer': q['well_formed'][0], 'gold': gold})
print(f'문장형 정답 질문 {len(wf_pool):,}개 / 답 없음 질문 {len(noans_pool):,}개 (모두 문단이 DB에 있음)')

rng = random.Random(SEED)
rng.shuffle(wf_pool); rng.shuffle(noans_pool)
n_ref = int(N_TRAIN * REFUSE_FRAC)
wf_pick    = wf_pool[:int((N_TRAIN - n_ref) * 1.5)]     # 정답 문단이 상위 5개 밖인 경우를 대비한 여유분
noans_pick = noans_pool[:n_ref]

t = time.time()
wf_docs    = retrieve_many([q['query'] for q in wf_pick])
noans_docs = retrieve_many([q['query'] for q in noans_pick]) if noans_pick else []
print(f'검색 {len(wf_pick) + len(noans_pick):,}개: {time.time() - t:.0f}초')

In [ ]:
def cite(answer, nums):
    tag = ''.join(f'[{n}]' for n in nums)
    a = answer.strip()
    return a[:-1] + f' {tag}.' if a.endswith('.') else f'{a} {tag}'

answerable, missed = [], []
for q, docs in zip(wf_pick, wf_docs):
    nums = [i for i, d in enumerate(docs, 1) if d['pid'] in q['gold']]
    ex = {'qid': q['qid'], 'query': q['query'], 'pids': [d['pid'] for d in docs]}
    (answerable if nums else missed).append({**ex, 'target': cite(q['answer'], nums) if nums else REFUSAL})
refusals = [{'qid': q['qid'], 'query': q['query'], 'pids': [d['pid'] for d in docs], 'target': REFUSAL}
            for q, docs in zip(noans_pick, noans_docs)]
refusals = (refusals + missed)[:n_ref]

train_ex = answerable[:N_TRAIN - len(refusals)] + refusals
random.Random(SEED).shuffle(train_ex)
save_jsonl(train_ex, f'{OUT_DIR}/train_examples.jsonl')
print(f'정답 문단이 상위 5개에 있음 {len(answerable):,} / 없음 {len(missed):,}')
print(f'학습 예시 {len(train_ex):,}개 (문장형 답변 {len(train_ex) - len(refusals):,} + 거절 {len(refusals):,})')
for ex in [e for e in train_ex if e['target'] != REFUSAL][:3]:
    print(' -', ex['query'], '→', ex['target'][:160])

## 6. LoRA 학습

05와 같은 설정입니다. 프롬프트(시스템 지시 + 문서 [1]~[5] + 질문)는 04·05와 똑같고, 답변 부분만 손실을 계산합니다.
체크포인트는 드라이브 `lora_sentence/ckpt/`에 저장됩니다. 런타임이 끊기면 다시 Run all 하면 이어서 학습합니다.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model

SYS_RAG = f'''You are a careful medical information assistant.
Rules:
1. Answer ONLY with facts stated in the numbered documents. Do not add outside knowledge.
2. After each sentence, cite the supporting document numbers like [1] or [2][4].
3. If the question is not about health or medicine, or the documents do not contain the answer, reply with exactly this sentence: "{REFUSAL}"
4. Answer in 2-4 short sentences.'''

def rag_messages(q, docs):
    ctx = '\n'.join(f'[{i}] ({d["site"]}) {d["text"]}' for i, d in enumerate(docs, 1))
    return [{'role': 'system', 'content': SYS_RAG},
            {'role': 'user', 'content': f'Documents:\n{ctx}\n\nQuestion: {q}'}]

tok = AutoTokenizer.from_pretrained(LLM_NAME)
llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, dtype=torch.bfloat16, device_map='cuda')
llm.config.use_cache = False
llm.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
llm.enable_input_require_grads()
llm = get_peft_model(llm, LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05, task_type='CAUSAL_LM',
                                     target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                                                     'gate_proj', 'up_proj', 'down_proj']))
llm.print_trainable_parameters()

def to_features(ex):
    docs = [{'site': sites[p], 'text': texts[p]} for p in ex['pids']]
    prompt = tok.apply_chat_template(rag_messages(ex['query'], docs), tokenize=False, add_generation_prompt=True)
    p_ids = tok(prompt, add_special_tokens=False)['input_ids']
    a_ids = tok(ex['target'] + '<|im_end|>', add_special_tokens=False)['input_ids']
    return {'input_ids': p_ids + a_ids, 'labels': [-100] * len(p_ids) + a_ids}

feats = [f for f in map(to_features, train_ex) if len(f['input_ids']) <= MAX_LEN]
print(f'학습 예시 {len(feats):,}개 / 평균 길이 {np.mean([len(f["input_ids"]) for f in feats]):.0f} 토큰')

def collate(batch):
    n = max(len(f['input_ids']) for f in batch)
    pad = lambda xs, v: [x + [v] * (n - len(x)) for x in xs]
    ids = pad([f['input_ids'] for f in batch], tok.pad_token_id)
    return {'input_ids': torch.tensor(ids),
            'attention_mask': torch.tensor(pad([[1] * len(f['input_ids']) for f in batch], 0)),
            'labels': torch.tensor(pad([f['labels'] for f in batch], -100))}

In [ ]:
CKPT_DIR = f'{OUT_DIR}/ckpt'
total_steps = -(-len(feats) // (BATCH * GRAD_ACC)) * EPOCHS
args = TrainingArguments(            # transformers 5: warmup_ratio 없음 → warmup_steps로 3%
    output_dir=CKPT_DIR, num_train_epochs=EPOCHS, learning_rate=LR, lr_scheduler_type='cosine',
    warmup_steps=max(1, int(total_steps * 0.03)),
    per_device_train_batch_size=BATCH, gradient_accumulation_steps=GRAD_ACC, bf16=True,
    logging_steps=10, save_steps=50, save_total_limit=2, report_to='none',
    remove_unused_columns=False, seed=SEED)
trainer = Trainer(model=llm, args=args, train_dataset=feats, data_collator=collate)

has_ckpt = os.path.isdir(CKPT_DIR) and any(d.startswith('checkpoint-') for d in os.listdir(CKPT_DIR))
t = time.time()
trainer.train(resume_from_checkpoint=has_ckpt or None)
print(f'학습 끝: {(time.time() - t) / 60:.0f}분')

ADAPTER_DIR = f'{OUT_DIR}/adapter'
llm.save_pretrained(ADAPTER_DIR)
tok.save_pretrained(ADAPTER_DIR)
pd.DataFrame([l for l in trainer.state.log_history if 'loss' in l]).to_csv(f'{OUT_DIR}/train_log.csv', index=False)
print('LoRA 저장:', ADAPTER_DIR)

## 7. 평가: 원본 Qwen vs 짧은 답 LoRA (05) vs 문장형 LoRA

세 시스템 모두 같은 검색 결과(상위 5개, 중복 제거)와 같은 거절 기준을 씁니다. 생성 모델만 다릅니다.
- **원본 Qwen**: LoRA를 잠시 끈 상태 (`disable_adapter`)
- **짧은 답 LoRA**: 05에서 v1.1 짧은 정답으로 학습한 LoRA를 불러와 사용
- **문장형 LoRA**: 이번에 학습한 LoRA

평가 질문: 문장형 정답이 있는 건강 질문(답 있음) + 답 없는 건강 질문 100 + 분야 밖 50

In [ ]:
import string

def toks(s):
    s = s.lower()
    s = re.sub(r'\[\d+\]', ' ', s)                          # 출처 번호는 채점에서 제외
    s = ''.join(ch if ch not in string.punctuation else ' ' for ch in s)
    return [w for w in s.split() if w not in {'a', 'an', 'the'}]

def f1(pred, ref):
    p, r = toks(pred), toks(ref)
    common = sum((Counter(p) & Counter(r)).values())
    if not p or not r or common == 0:
        return 0.0
    prec, rec = common / len(p), common / len(r)
    return 2 * prec * rec / (prec + rec)

def rouge_l(pred, ref):
    p, r = toks(pred), toks(ref)
    if not p or not r:
        return 0.0
    dp = [0] * (len(r) + 1)
    for a in p:
        prev = 0
        for j, b in enumerate(r, 1):
            cur = dp[j]
            dp[j] = prev + 1 if a == b else max(dp[j], dp[j - 1])
            prev = cur
    lcs = dp[-1]
    if lcs == 0:
        return 0.0
    prec, rec = lcs / len(p), lcs / len(r)
    return 2 * prec * rec / (prec + rec)

best = lambda fn, pred, refs: max(fn(pred, r) for r in refs)
has_cite = lambda a: bool(re.search(r'\[\d\]', a))

In [ ]:
def recall(pred, ref):
    # 정답 포함률: 짧은 정답의 단어 중 답변에 들어 있는 비율 (답변이 길어도 깎이지 않음)
    p, r = Counter(toks(pred)), Counter(toks(ref))
    return sum((p & r).values()) / max(1, sum(r.values()))

def is_refusal(ans):
    a = ans.lower()
    return any(p in a for p in ["can't answer", 'cannot answer', 'unable to answer'])

llm.eval()
llm.config.use_cache = True
tok.padding_side = 'left'

@torch.inference_mode()
def generate(msgs_list, bs=GEN_BATCH):
    outs = []
    for i in range(0, len(msgs_list), bs):
        prompts = [tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in msgs_list[i:i + bs]]
        enc = tok(prompts, return_tensors='pt', padding=True).to(llm.device)
        gen = llm.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False, temperature=None, top_p=None, top_k=None,
                           pad_token_id=tok.pad_token_id)
        outs += tok.batch_decode(gen[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)
    return [o.strip() for o in outs]

test_a = [q for q in eval_set if q['domain'] == 'health' and q['answerable'] and q['well_formed']]
test_n = [q for q in eval_set if q['domain'] == 'health' and not q['answerable']]
test_o = [q for q in eval_set if q['domain'] != 'health']
qs = test_a + test_n + test_o
queries = [q['query'] for q in qs]
print(f'평가: 문장형 정답 있는 건강 {len(test_a)} + 답 없음 {len(test_n)} + 분야 밖 {len(test_o)}')

t = time.time()
docs_all = retrieve_many(queries)
todo = [i for i, d in enumerate(docs_all) if d[0]['top_raw'] >= THRESH]
msgs = [rag_messages(queries[i], docs_all[i]) for i in todo]
retr_sec = (time.time() - t) / len(qs)
print(f'검색 끝. 점수 기준 거절 {len(qs) - len(todo)}개')

has_short = os.path.isdir(SHORT_LORA)
if has_short:
    llm.load_adapter(SHORT_LORA, adapter_name='short')

systems = [('원본 Qwen', None)] + ([('짧은 답 LoRA (05)', 'short')] if has_short else []) + [('문장형 LoRA', 'default')]
runs = {}
for name, adapter in systems:
    t = time.time()
    if adapter is None:
        with llm.disable_adapter():
            gen = generate(msgs)
    else:
        llm.set_adapter(adapter)
        gen = generate(msgs)
    answers = [REFUSAL] * len(qs)                     # 관련 문서 없음 → LLM 안 부르고 거절
    for i, a in zip(todo, gen):
        answers[i] = a
    runs[name] = {'answers': answers, 'sec': (time.time() - t) / len(qs) + retr_sec}
    print(f"{name} 끝 ({runs[name]['sec']:.2f}초/질문)")
llm.set_adapter('default')

In [ ]:
na, nn = len(test_a), len(test_n)
A_idx = range(na)
N_idx = range(na, na + nn)
O_idx = range(na + nn, len(qs))
rows = {}
for name, r in runs.items():
    A = r['answers']
    answered = [i for i in A_idx if not is_refusal(A[i])]
    rows[name] = {
        'ROUGE-L (문장형)': np.mean([best(rouge_l, A[i], qs[i]['well_formed']) for i in A_idx]),
        '정답 포함률':     np.mean([best(recall, A[i], qs[i]['answers']) for i in A_idx]),
        'F1 (짧은 정답)':  np.mean([best(f1, A[i], qs[i]['answers']) for i in A_idx]),
        '출처 표시율':     np.mean([has_cite(A[i]) for i in answered]) if answered else np.nan,
        '잘못 거절':       np.mean([is_refusal(A[i]) for i in A_idx]),
        '답 없음 거절률':  np.mean([is_refusal(A[i]) for i in N_idx]),
        '분야 밖 거절률':  np.mean([is_refusal(A[i]) for i in O_idx]),
        '평균 답변 단어':  np.mean([len(A[i].split()) for i in answered]) if answered else np.nan,
        '초/질문':        r['sec'],
    }
table = pd.DataFrame(rows).T.round(3)
r5 = np.mean([bool(set(qs[i]['gold_pids']) & {d['pid'] for d in docs_all[i]}) for i in A_idx])
display(table)
print(f'근거 R@5 (답 있는 건강 질문): {r5:.3f}')

summary = '\n'.join([
    f"# 06 문장형 답변 LoRA (v2.1) ({time.strftime('%Y-%m-%d %H:%M', time.localtime(time.time() + 9 * 3600))} KST)", '',
    f'- 학습: v2.1 train 건강 질문 {len(feats):,}개 (문장형 답변 {len(train_ex) - len(refusals):,} + 거절 {len(refusals):,}), '
    f'LoRA r={LORA_R}, {EPOCHS} epoch, {LLM_NAME}',
    f'- DB: v2.1 문단 {len(corpus):,}개 / 검색: bge → MiniLM-L-6 상위 {TOP_N}개, 중복 제거 / 거절 기준(검증 하위 5%) = {THRESH:.2f}',
    f'- 평가: 문장형 정답 있는 건강 {na} + 답 없음 {nn} + 분야 밖 {len(test_o)} / 근거 R@5 = {r5:.3f}',
    '- ROUGE-L (문장형): 문장형 정답과 비교 / 정답 포함률: 짧은 정답 단어가 답변에 들어 있는 비율', '',
    table.to_markdown(),
])
for d in (OUT_DIR, REPORT_DIR):
    with open(f'{d}/results_summary.md', 'w', encoding='utf-8') as f:
        f.write(summary)
    save_jsonl([{'qid': q['qid'], 'domain': q['domain'], 'answerable': q['answerable'], 'query': q['query'],
                 'gold_answer': (q.get('answers') or [''])[0], 'well_formed': (q.get('well_formed') or [''])[0],
                 **{name: r['answers'][i] for name, r in runs.items()},
                 'docs': [{k: d[k] for k in ('pid', 'score', 'site', 'tier')} for d in docs_all[i]]}
                for i, q in enumerate(qs)], f'{d}/eval_answers.jsonl')
pd.read_csv(f'{OUT_DIR}/train_log.csv').to_csv(f'{REPORT_DIR}/train_log.csv', index=False)
print('저장:', OUT_DIR, '/', REPORT_DIR)

In [ ]:
# 답변 비교 예시 (번호를 바꿔 보세요. 앞쪽은 답 있는 건강 질문)
i = 0
print('질문:', qs[i]['query'])
print('문장형 정답:', (qs[i].get('well_formed') or ['-'])[0][:200])
print('짧은 정답 :', (qs[i].get('answers') or ['-'])[0][:200])
for name, r in runs.items():
    print(f'\n[{name}]', r['answers'][i])

## 8. 드라이브 저장 마무리 + 런타임 종료

문장형 LoRA는 드라이브 `health/lora_sentence/adapter/`에 있습니다. 04 챗봇에서 쓰려면 LLM을 불러온 뒤 `PeftModel.from_pretrained(llm, ADAPTER_DIR)` 한 줄을 넣으면 됩니다.

In [ ]:
from google.colab import drive, runtime
drive.flush_and_unmount()
runtime.unassign()